In [ ]:
%pip install -q requests pandas

In [ ]:
import json
import sys
from pathlib import Path

import requests
import pandas as pd

OLLAMA_URL = "http://localhost:11434"
MODEL = "gemma2:9b"

SEARCH_ROOTS = [
    Path("results"),
    Path("../results"),
    Path("/content/drive/MyDrive/DhwaniLab/results"),
]

OIWER_ROOTS = [Path("oiwer"), Path("../oiwer")]

for root in OIWER_ROOTS:
    if (root / "oiwer.py").exists():
        sys.path.insert(0, str(root.resolve()))
        break

from oiwer import score_utterance, normalize_segments, Counts

print("oiwer module loaded")

In [ ]:
try:
    tags = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()
    installed = [m["name"] for m in tags.get("models", [])]
    print("Ollama reachable")
    print("Installed models:", installed)
    print()
    print("Selected model:", MODEL)
    print("Available:", MODEL in installed)
except Exception as e:
    print("Ollama NOT reachable:", type(e).__name__, e)
    print()
    print("Install from https://ollama.com/download then run:")
    print(f"  ollama pull {MODEL}")
    print("  ollama serve")

In [ ]:
MODELS = ["indicwhisper", "indicconformer"]

def resolve(model_name):
    for root in SEARCH_ROOTS:
        path = root / model_name / "indicvoices_telugu_valid.csv"
        if path.exists():
            return path
    return None

frames = {}

for model_name in MODELS:
    path = resolve(model_name)
    if path is None:
        raise FileNotFoundError(f"No CSV for {model_name}")
    df = pd.read_csv(path)
    df["index"] = df["index"].astype(int)
    df["reference"] = df["reference"].fillna("")
    df["prediction"] = df["prediction"].fillna("")
    frames[model_name] = df
    print(f"{model_name:16} {path}  rows={len(df)}")

references = (
    frames[MODELS[0]][["index", "reference"]]
    .sort_values("index")
    .reset_index(drop=True)
)

print()
print("Unique references to process:", len(references))

In [ ]:
PROMPT = """You are a Telugu language expert helping build an ASR evaluation benchmark.

Given a Telugu sentence, list the acceptable alternative spellings for each word.
Two spellings are alternatives ONLY if a fluent Telugu reader would accept both as
correct writings of the same spoken word.

Consider these variation types:
1. Matra and diacritic variations
2. Alternative spellings of loanwords from English, Sanskrit, Urdu
3. A compound word written as one word or split into two
4. Regional or dialectal spellings
5. Ligature written joined or separate
6. Sandhi merging or splitting at word boundaries
7. Numbers written as digits or as words

Rules:
- Output ONLY valid JSON, no explanation.
- Format: {{"segments": [["original", "variant", ...], ["original"], ...]}}
- The FIRST entry of each segment must be the original text exactly as given.
- Segments must cover every word of the sentence, in order, with nothing omitted.
- If a word has no acceptable alternative, output a segment containing only that word.
- To express merging two words, put both words in one segment as "word1 word2".
- Do NOT invent spellings you are unsure about. Fewer, correct variants are better.

Sentence: {sentence}"""


def generate_segments(sentence, model=MODEL, timeout=120):
    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": model,
            "prompt": PROMPT.format(sentence=sentence),
            "format": "json",
            "stream": False,
            "options": {"temperature": 0.2},
        },
        timeout=timeout,
    )
    response.raise_for_status()

    payload = json.loads(response.json()["response"])
    return payload.get("segments", [])


def validate(segments, reference):
    if not segments:
        return None

    canonical = []
    for slot in segments:
        if not isinstance(slot, list) or not slot:
            return None
        canonical.extend(str(slot[0]).split())

    if canonical != reference.split():
        return None

    return normalize_segments(segments)


def trivial_segments(reference):
    return [[[token]] for token in reference.split()]

In [ ]:
SAMPLE_SIZE = 15

sample_rows = references.head(SAMPLE_SIZE).to_dict("records")

sample_results = []

for row in sample_rows:
    reference = row["reference"]
    try:
        raw = generate_segments(reference)
        validated = validate(raw, reference)
        status = "OK" if validated else "REJECTED"
    except Exception as e:
        raw, validated, status = None, None, f"ERROR {type(e).__name__}"

    sample_results.append({
        "index": row["index"],
        "reference": reference,
        "raw": raw,
        "validated": validated,
        "status": status,
    })

    print(f"[{row['index']}] {status}")

ok = sum(1 for r in sample_results if r["status"] == "OK")
print()
print(f"Valid outputs: {ok}/{len(sample_results)}")

In [ ]:
for r in sample_results:
    if r["status"] != "OK":
        continue

    varied = [s for s in r["validated"] if len(s) > 1]

    if not varied:
        continue

    print("REFERENCE:", r["reference"])
    for slot in varied:
        print("   ", " | ".join(" ".join(v) for v in slot))
    print()

In [ ]:
CACHE_FILE = Path("oiwer_variants_te.json")

if CACHE_FILE.exists():
    cache = json.loads(CACHE_FILE.read_text(encoding="utf-8"))
else:
    cache = {}

print("Cached utterances:", len(cache))
print("Remaining:", len(references) - len(cache))

In [ ]:
SAVE_EVERY = 25

rows = references.to_dict("records")
processed = 0
rejected = 0

for row in rows:
    key = str(row["index"])

    if key in cache:
        continue

    reference = row["reference"]

    try:
        raw = generate_segments(reference)
        validated = validate(raw, reference)
    except Exception as e:
        print(f"[{key}] ERROR {type(e).__name__}")
        validated = None

    if validated is None:
        rejected += 1
        validated = trivial_segments(reference)

    cache[key] = validated
    processed += 1

    if processed % SAVE_EVERY == 0:
        CACHE_FILE.write_text(
            json.dumps(cache, ensure_ascii=False), encoding="utf-8"
        )
        print(f"{len(cache)}/{len(rows)} cached | rejected so far: {rejected}")

CACHE_FILE.write_text(json.dumps(cache, ensure_ascii=False), encoding="utf-8")

print()
print("Done. Cached:", len(cache))
print("Fell back to trivial segments:", rejected)

In [ ]:
cache = json.loads(CACHE_FILE.read_text(encoding="utf-8"))

with_variants = 0
total_slots = 0
varied_slots = 0

for key, segments in cache.items():
    has = False
    for slot in segments:
        total_slots += 1
        if len(slot) > 1:
            varied_slots += 1
            has = True
    if has:
        with_variants += 1

print("Utterances with at least one variant:", with_variants, "/", len(cache))
print("Slots with a variant:", varied_slots, "/", total_slots)
print(f"Variant coverage: {100 * varied_slots / max(total_slots, 1):.2f}% of words")

In [ ]:
results = []

for model_name, df in frames.items():
    wer_total = Counts()
    oiwer_total = Counts()

    for row in df.to_dict("records"):
        key = str(row["index"])
        segments = cache.get(key)

        wer_total = wer_total + score_utterance(
            row["reference"], row["prediction"]
        )
        oiwer_total = oiwer_total + score_utterance(
            row["reference"], row["prediction"], segments
        )

    results.append({
        "model": model_name,
        "samples": len(df),
        "reference_words": wer_total.reference_words,
        "wer_percent": wer_total.wer * 100,
        "oiwer_percent": oiwer_total.wer * 100,
        "improvement": (wer_total.wer - oiwer_total.wer) * 100,
        "wer_sub": wer_total.substitutions,
        "oiwer_sub": oiwer_total.substitutions,
        "sub_removed": wer_total.substitutions - oiwer_total.substitutions,
    })

comparison_df = pd.DataFrame(results).sort_values("oiwer_percent")

comparison_df

In [ ]:
for row in comparison_df.to_dict("records"):
    print(f"{row['model']:16} "
          f"WER {row['wer_percent']:6.2f}%  ->  "
          f"OIWER {row['oiwer_percent']:6.2f}%  "
          f"(improved {row['improvement']:.2f} pts, "
          f"{row['sub_removed']} substitutions removed)")

print()

best = comparison_df.iloc[0]
worst = comparison_df.iloc[-1]

wer_gap = worst["wer_percent"] - best["wer_percent"]
oiwer_gap = worst["oiwer_percent"] - best["oiwer_percent"]

print(f"Model gap under WER  : {wer_gap:.2f} points")
print(f"Model gap under OIWER: {oiwer_gap:.2f} points")

In [ ]:
output_dir = Path("results/oiwer")
output_dir.mkdir(parents=True, exist_ok=True)

comparison_df.to_csv(output_dir / "oiwer_comparison.csv", index=False)

print("Saved:", output_dir / "oiwer_comparison.csv")
print()
print(comparison_df.to_string(index=False))